# Testing the `Character` class

## Setup

In [1]:
import notebook_setup
%preparse gaknot
from gaknot.gaknot import GeneralizedAlgebraicKnot
%preparse H1_branched_cover
from gaknot.H1_branched_cover import BranchedCoverHomology
%preparse character
from gaknot.character import Character
from sage.all import QQ, ZZ
import ipytest
import pytest

ipytest.autoconfig()

Notebook setup complete. Environment configured.
INFO: 

import_sage called with arguments:
	module_name: gaknot
	package: gaknot
	path: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function
Successfully preparsed and reloaded: gaknot
INFO: 

import_sage called with arguments:
	module_name: H1_branched_cover
	package: gaknot
	path: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function
Successfully preparsed and reloaded: H1_branched_cover
INFO: 

import_sage called with arguments:
	module_name: character
	package: gaknot
	path: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function
Successfully preparsed and reloaded: character


## Tests

### Test Case 1: Simple Torus Knots (Parametric - 10 Cases)

**Subject:** Simple Torus Knots $T(p,q)$ at varying covering degrees $N$.

In [2]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("p, q, N, expected_factors, char_values", [
    (2, 3, 2, [3],       [1/3]),          # 1. Trefoil (Z/3)
    (2, 5, 2, [5],       [1/5]),          # 2. T(2,5) (Z/5)
    (2, 7, 2, [7],       [1/7]),          # 3. T(2,7) (Z/7)
    (2, 3, 3, [2, 2],    [1/2, 1/2]),     # 4. T(2,3) at N=3 (Z/2 + Z/2)
    (3, 4, 2, [3],       [1/3]),          # 5. T(3,4) (Z/3)
    (3, 5, 2, [],        []),             # 6. T(3,5) (Trivial H1)
    (2, 5, 4, [5],       [1/5]),          # 7. T(2,5) at N=4 (Z/5) - Corrected
    (2, 11, 2, [11],     [1/11]),         # 8. T(2,11) (Z/11)
    (2, 31, 2, [31],     [1/31]),         # 9. T(2,31) (Z/31)
    (3, 2, 4, [3],       [1/3])           # 10. T(3,2) at N=4 (Z/3) - Corrected
])
def test_case_1_simple_torus_parametric(p, q, N, expected_factors, char_values):
    # 1. Setup Knot
    knot = GeneralizedAlgebraicKnot([(1, [(p, q)])])
    homology = BranchedCoverHomology(knot, N)
    
    # 2. Verify Homology Factors
    actual_factors = [f for f in homology.invariant_factors if f > 1]
    assert actual_factors == expected_factors
    
    # 3. Define Character
    nested_values = [ [char_values] ]
    chi = Character(homology, nested_values)
    
    # 4. Assertions
    layer_vals = chi.restrict_to_layer(0, 0)
    assert layer_vals == [char_values]

======================================= test session starts ========================================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/sage_env/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/notebooks
plugins: anyio-4.12.1, nbmake-1.5.5, langsmith-0.7.3
collecting ... collected 10 items

t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_case_1_simple_torus_parametric[p0-q0-N0-expected_factors0-char_values0] PASSED [ 10%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_case_1_simple_torus_parametric[p1-q1-N1-expected_factors1-char_values1] PASSED [ 20%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_case_1_simple_torus_parametric[p2-q2-N2-expected_factors2-char_values2] PASSED [ 30%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_case_1_simple_torus_parametric[p3-q3-N3-expected_factors3-char_values3] PASSED [ 40%]
t_bfd7b11409914a0

### Test Case 2: Degenerate Satellite (Trivial Outer Layer)

**Subject:** Satellite Knots where one layer contributes no generators.

In [3]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("cable, N, outer_factors, inner_factors, inner_values", [
    ([(2, 3), (2, 5)], 3, [], [2, 2], [1/2, 1/2]),
    ([(2, 5), (3, 5)], 2, [], [5], [1/5]),
    ([(3, 4), (3, 5)], 2, [], [3], [1/3]),
    ([(2, 3), (3, 7)], 2, [], [3], [1/3]),
    ([(2, 11), (2, 3)], 3, [2, 2], [], []),
    ([(2, 3), (2, 5)], 5, [2, 2, 2, 2], [], []),
    ([(3, 2), (2, 3)], 2, [3], [], []),
    ([(2, 5), (2, 7)], 2, [7], [], []),
    ([(2, 3), (2, 11)], 2, [11], [], []),
    ([(2, 7), (2, 5)], 3, [], [], [])
])
def test_case_2_degenerate_satellite(cable, N, outer_factors, inner_factors, inner_values):
    knot = GeneralizedAlgebraicKnot([(1, cable)])
    homology = BranchedCoverHomology(knot, N)
    
    # Verify structure matches expectations
    # layers[0] is outer, layers[1] is inner
    assert homology.decomposition[0]["layers"][0]["base_factors"] == outer_factors
    assert homology.decomposition[0]["layers"][1]["base_factors"] == inner_factors
    
    # Define Character
    # Note: Multiplicity check
    m_outer = homology.decomposition[0]["layers"][0]["multiplicity"]
    m_inner = homology.decomposition[0]["layers"][1]["multiplicity"]
    
    outer_vals_input = [0] * (m_outer * len(outer_factors))
    inner_vals_input = inner_values # Assumed to be provided with multiplicity in mind or simple
    # Fix for multiplicity: if inner_values is just factors, repeat it
    if len(inner_vals_input) != m_inner * len(inner_factors):
        inner_vals_input = inner_values * m_inner
        
    values = [ [ outer_vals_input, inner_vals_input ] ]
    chi = Character(homology, values)
    
    # Assertions
    assert chi.restrict_to_layer(0, 0) == [ [0]*len(outer_factors) ] * m_outer
    assert chi.restrict_to_layer(0, 1) == [ inner_values ] * m_inner


### Test Case 3: Fully Nontrivial Satellite

**Subject:** Satellite Knots where both layers contribute generators.

In [4]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("cable, N, outer_factors, inner_factors", [
    ([(2, 5), (3, 2)], 2, [3], [5]),
    ([(2, 3), (2, 7)], 2, [7], [3]),
    ([(3, 2), (2, 5)], 2, [5], [3]),
    ([(2, 5), (2, 3)], 2, [3], [5]),
    ([(2, 3), (3, 4)], 2, [3], [3]),
    ([(2, 7), (2, 3)], 2, [3], [7]),
    ([(2, 5), (2, 11)], 2, [11], [5]),
    ([(2, 11), (2, 5)], 2, [5], [11]),
    ([(3, 4), (2, 3)], 2, [3], [3]),
    ([(2, 3), (2, 5)], 2, [5], [3])
])
def test_case_3_nontrivial_cable_full(cable, N, outer_factors, inner_factors):
    knot = GeneralizedAlgebraicKnot([(1, cable)])
    homology = BranchedCoverHomology(knot, N)
    
    # Map each generator to 1/modulus
    o_vals = [1/f for f in outer_factors]
    i_vals = [1/f for f in inner_factors]
    
    values = [ [ o_vals, i_vals ] ]
    chi = Character(homology, values)
    
    assert chi.restrict_to_layer(0, 0) == [o_vals]
    assert chi.restrict_to_layer(0, 1) == [i_vals]

======================================= test session starts ========================================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/sage_env/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/notebooks
plugins: anyio-4.12.1, nbmake-1.5.5, langsmith-0.7.3
collecting ... collected 10 items

t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_case_3_nontrivial_cable_full[cable0-N0-outer_factors0-inner_factors0] PASSED [ 10%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_case_3_nontrivial_cable_full[cable1-N1-outer_factors1-inner_factors1] FAILED [ 20%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_case_3_nontrivial_cable_full[cable2-N2-outer_factors2-inner_factors2] FAILED [ 30%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_case_3_nontrivial_cable_full[cable3-N3-outer_factors3-inner_factors3] FAILED [ 40%]
t_bfd7b11409914a03a1d6b47

### Test Case 4: Connected Sum (Geometric vs. Sorted Order)

In [5]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("p1, q1, p2, q2, N", [
    (5, 2, 3, 2, 2),
    (7, 2, 3, 2, 2),
    (11, 2, 5, 2, 2),
    (3, 2, 5, 2, 2),
    (5, 2, 7, 2, 2),
    (2, 3, 2, 5, 3),
    (2, 5, 2, 3, 2),
    (3, 4, 2, 3, 2),
    (2, 3, 3, 4, 2),
    (2, 11, 2, 3, 2)
])
def test_case_4_connected_sum_geometric_order(p1, q1, p2, q2, N):
    k1 = GeneralizedAlgebraicKnot([(1, [(p1, q1)])])
    k2 = GeneralizedAlgebraicKnot([(1, [(p2, q2)])])
    knot_sum = k1 + k2
    homology = BranchedCoverHomology(knot_sum, N)
    
    f1 = [f for f in BranchedCoverHomology(k1, N).invariant_factors if f > 1]
    f2 = [f for f in BranchedCoverHomology(k2, N).invariant_factors if f > 1]
    
    v1 = [1/f for f in f1]
    v2 = [1/f for f in f2]
    
    values = [ [v1], [v2] ]
    chi = Character(homology, values)
    
    assert chi.restrict_to_layer(0, 0) == [v1]
    assert chi.restrict_to_layer(1, 0) == [v2]

======================================= test session starts ========================================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/sage_env/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/notebooks
plugins: anyio-4.12.1, nbmake-1.5.5, langsmith-0.7.3
collecting ... collected 10 items

t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_case_4_connected_sum_geometric_order[p10-q10-p20-q20-N0] PASSED [ 10%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_case_4_connected_sum_geometric_order[p11-q11-p21-q21-N1] PASSED [ 20%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_case_4_connected_sum_geometric_order[p12-q12-p22-q22-N2] PASSED [ 30%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_case_4_connected_sum_geometric_order[p13-q13-p23-q23-N3] PASSED [ 40%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_case_4_connected_sum_geometric_or

### Test Case 5: Modulus Compatibility Validation

In [6]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("p, q, N, bad_val", [
    (2, 3, 3, 1/3), # H1 = Z/2 + Z/2
    (2, 5, 2, 1/3), # H1 = Z/5
    (2, 3, 2, 1/2), # H1 = Z/3
    (3, 4, 2, 1/2), # H1 = Z/3
    (2, 7, 2, 1/5), # H1 = Z/7
    (2, 5, 4, 1/2), # H1 = Z/5
    (2, 11, 2, 1/3), # H1 = Z/11
    (2, 3, 3, 1/7), # H1 = Z/2 + Z/2
    (3, 2, 4, 1/2), # H1 = Z/3
    (2, 31, 2, 1/2)  # H1 = Z/31
])
def test_validation_modulus_compatibility(p, q, N, bad_val):
    knot = GeneralizedAlgebraicKnot([(1, [(p, q)])])
    homology = BranchedCoverHomology(knot, N)
    factors = [f for f in homology.invariant_factors if f > 1]
    
    if not factors:
        pytest.skip("Trivial homology")
        
    bad_values = [ [ [bad_val] * len(factors) ] ]
    with pytest.raises(ValueError, match="not compatible with Z"):
        Character(homology, bad_values)

======================================= test session starts ========================================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/sage_env/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/notebooks
plugins: anyio-4.12.1, nbmake-1.5.5, langsmith-0.7.3
collecting ... collected 10 items

t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_validation_modulus_compatibility[p0-q0-N0-bad_val0] PASSED [ 10%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_validation_modulus_compatibility[p1-q1-N1-bad_val1] PASSED [ 20%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_validation_modulus_compatibility[p2-q2-N2-bad_val2] PASSED [ 30%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_validation_modulus_compatibility[p3-q3-N3-bad_val3] PASSED [ 40%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_validation_modulus_compatibility[p4-q4-N4-bad_val4] P

### Test Case 6: Heterogeneous Connected Sum (Simple + Satellite)

In [7]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("simple_pq, satellite_cable, N", [
    ((2, 5), [(2, 5), (2, 3)], 2),
    ((2, 3), [(2, 3), (2, 5)], 2),
    ((3, 2), [(2, 5), (3, 2)], 2),
    ((2, 7), [(2, 3), (2, 7)], 2),
    ((2, 3), [(2, 5), (2, 3)], 2),
    ((2, 5), [(2, 7), (2, 5)], 2),
    ((2, 11), [(2, 3), (2, 11)], 2),
    ((3, 4), [(2, 3), (3, 4)], 2),
    ((2, 5), [(3, 2), (2, 5)], 2),
    ((2, 3), [(2, 11), (2, 3)], 2)
])
def test_heterogeneous_connected_sum(simple_pq, satellite_cable, N):
    k1 = GeneralizedAlgebraicKnot([(1, [simple_pq])])
    k2 = GeneralizedAlgebraicKnot([(1, satellite_cable)])
    knot_sum = k1 + k2
    homology = BranchedCoverHomology(knot_sum, N)
    
    # Get actual factors for values assignment
    h1 = BranchedCoverHomology(k1, N)
    h2 = BranchedCoverHomology(k2, N)
    
    v1 = [1/f for f in h1.invariant_factors if f > 1]
    
    # Layer-wise values for satellite
    v_sat = []
    for layer in h2.decomposition[0]['layers']:
        m = layer['multiplicity']
        f = layer['base_factors']
        v_sat.append([1/x for x in f] * m)
        
    values = [ [v1], v_sat ]
    chi = Character(homology, values)
    
    assert chi.restrict_to_layer(0, 0) == [v1] if v1 else [[]]
    for i, layer_vals in enumerate(chi.restrict_to_layer(1, i) for i in range(len(v_sat))):
        expected = [ [1/x for x in h2.decomposition[0]['layers'][i]['base_factors']] ] * h2.decomposition[0]['layers'][i]['multiplicity']
        assert layer_vals == expected

======================================= test session starts ========================================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/sage_env/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/notebooks
plugins: anyio-4.12.1, nbmake-1.5.5, langsmith-0.7.3
collecting ... collected 10 items

t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_heterogeneous_connected_sum[simple_pq0-satellite_cable0-N0] PASSED [ 10%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_heterogeneous_connected_sum[simple_pq1-satellite_cable1-N1] PASSED [ 20%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_heterogeneous_connected_sum[simple_pq2-satellite_cable2-N2] PASSED [ 30%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_heterogeneous_connected_sum[simple_pq3-satellite_cable3-N3] PASSED [ 40%]
t_bfd7b11409914a03a1d6b476d04e7ec1.py::test_heterogeneous_connect